In [ ]:
#yfinance doesn't access articles, so im testing sec edgar api

In [1]:
import requests

def get_cik(ticker):
    url = f"https://www.sec.gov/files/company_tickers.json"
    data = requests.get(url).json()
    for k, v in data.items():
        if v['ticker'].lower() == ticker.lower():
            return v['cik_str'].zfill(10)
    return None

def get_filings(cik, form_type="10-K", count=5):
    headers = {'User-Agent': 'Your Name contact@yourdomain.com'}
    base_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    r = requests.get(base_url, headers=headers)
    data = r.json()
    filings = data['filings']['recent']

    result = []
    for i in range(len(filings['form'])):
        if filings['form'][i] == form_type:
            report = {
                'form': filings['form'][i],
                'date': filings['filingDate'][i],
                'url': f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{filings['accessionNumber'][i].replace('-', '')}/{filings['primaryDocument'][i]}"
            }
            result.append(report)
        if len(result) >= count:
            break
    return result

# Example
ticker = "MSFT"
cik = get_cik(ticker)
filings = get_filings(cik, "10-K")
for f in filings:
    print(f"{f['date']} - {f['form']} - {f['url']}")


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import requests

def get_cik(ticker):
    url = "https://www.sec.gov/files/company_tickers.json"
    
    # Check status of the request
    response = requests.get(url)
    print(f"Status Code: {response.status_code}")
    print(f"Response Text: {response.text[:100]}...")  # Preview the first 100 chars

    if response.status_code != 200:
        raise ValueError("Failed to retrieve data from SEC API.")
    
    # Try parsing JSON
    data = response.json()
    for k, v in data.items():
        if v['ticker'].lower() == ticker.lower():
            return v['cik_str'].zfill(10)
    return None

# Test with MSFT
ticker = "MSFT"
cik = get_cik(ticker)
if cik:
    print(f"CIK for {ticker}: {cik}")
else:
    print("CIK not found.")


In [ ]:
# beautiful soup is being used to scrape yahoo finance of headlines

In [10]:
import requests
from bs4 import BeautifulSoup

def get_sec_filings(cik, form_type="10-K"):
    url = f"https://www.sec.gov/edgar/searchedgar/companysearch.html?CIK={cik}&type={form_type}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    filings = []
    for row in soup.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) > 3:
            filing_date = cols[3].text.strip()
            filing_url = "https://www.sec.gov" + cols[1].find('a')['href']
            filings.append({"date": filing_date, "url": filing_url})
    
    return filings

# Test for Microsoft CIK (0000789019)
cik = "0000789019"
filings = get_sec_filings(cik)

if filings:
    for filing in filings[:5]:  # Get top 5 filings
        print(f"Filing Date: {filing['date']}")
        print(f"Filing URL: {filing['url']}")
        print("---")
else:
    print("No filings found for Microsoft.")
    
# Test for Apple CIK (0000320193)
cik_apple = "0000320193"
filings_apple = get_sec_filings(cik_apple)

if filings_apple:
    for filing in filings_apple[:5]:  # Get top 5 filings
        print(f"Filing Date: {filing['date']}")
        print(f"Filing URL: {filing['url']}")
        print("---")
else:
    print("No filings found for Apple.")



No filings found for Microsoft.
No filings found for Apple.


In [ ]:
#other sources that could be used: finnhub.io (Free tier, news, and SEC filings), 
#financialmodelingprep.com (SEC filings + press)